Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
silver_table = f"{catalog}.{silver_schema}.stations"
gold_table = f"{catalog}.{gold_schema}.dim_airports"

Read silver Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

silver_stations_df=(
    spark.read
    .format("delta")
    .table(silver_table)
    .filter(F.col("batch_id")==batch_id)
)

Select and rename columns for clarity and unification to create dimension table

In [0]:
gold_dim_airports_df=(
    silver_stations_df
    .withColumnsRenamed({
        "airport_id" : "airport_key",
        "airport" : "airport_id"
    })
)

In [0]:
gold_dim_airports_df=(
    gold_dim_airports_df
    .select(
        "airport_key",
        "airport_id",
        "display_airport_name",
        "airport_city",
        "airport_state",
        "airport_state_code",
        "latitude",
        "longitude",
        "elevation",
        "icao",
        "iata",
        "faa",
        "mesonet_station"

    )
)

In [0]:
from pyspark.sql import functions as F

(
    gold_dim_airports_df
    .groupBy("airport_key")
    .count()
    .filter(F.col("count") > 1)
    .show(truncate=False)
)

+-----------+-----+
|airport_key|count|
+-----------+-----+
|14893      |2    |
|12255      |2    |
|14492      |2    |
|13832      |2    |
|11066      |2    |
|14307      |2    |
|14457      |2    |
|10423      |2    |
|13829      |2    |
|14794      |2    |
|12264      |2    |
|15024      |2    |
|15991      |2    |
|15401      |2    |
|11695      |2    |
|10800      |2    |
|10561      |2    |
|14288      |2    |
|11624      |2    |
|14082      |2    |
+-----------+-----+
only showing top 20 rows


Write DataFrame to silver Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
if not spark.catalog.tableExists(gold_table):
    gold_dim_airports_df_write=(
        gold_dim_airports_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, gold_table)
    (
        delta_table.alias("t")
        .merge(
            gold_dim_airports_df.alias("s"),
            "t.airport_key = s.airport_key"
        )
        .whenMatchedUpdate(
            set={
                "airport_key": "s.airport_key",
                "airport_id" :"s.airport_id",
                "display_airport_name" : "s.display_airport_name",
                "airport_city": "s.airport_city",
                "airport_state" : "s.airport_state",
                "airport_state_code" : "s.airport_state_code",
                "latitude": "s.latitude",
                "longitude": "s.longitude",
                "elevation" : "s.elevation",
                "icao": "s.icao",
                "iata": "s.iata",
                "faa": "s.faa",
                "mesonet_station": "s.mesonet_station",
            }

        )
        .whenNotMatchedInsertAll()
        .execute()
    )

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8385385331910283>, line 39
     11 from delta.tables import DeltaTable
     13 delta_table=DeltaTable.forName(spark, gold_table)
     14 (
     15     delta_table.alias("t")
     16     .merge(
     17         gold_dim_airports_df.alias("s"),
     18         "t.airport_key = s.airport_key"
     19     )
     20     .whenMatchedUpdate(
     21         set={
     22             "airport_key": "s.airport_key",
     23             "airport_id" :"s.airport_id",
     24             "display_airport_name" : "s.display_airport_name",
     25             "airport_city": "s.airport_city",
     26             "airport_state" : "s.airport_state",
     27             "airport_state_code" : "s.airport_state_code",
     28             "latitude": "s.latitude",
     29             "longitude": "s.longitude",
     30             "elevation